# Анализ датасета EdStatsCountry
**Источник данных:** `EdStatsCountry.csv` — статистика по образованию и экономике стран мира (Всемирный банк)

---
Ноутбук выполняет:
- Загрузку и первичный осмотр данных
- Корреляционный анализ числовых колонок
- Статистику (среднее, отклонение, MAE) по каждой числовой колонке
- Анализ категориальных колонок
- Сводные таблицы по регионам и группам дохода

## 1. Самописные статистические функции

In [ ]:
def meanvalue(data):
    """Среднее арифметическое"""
    return sum(data) / len(data)


def meandeviation(data):
    """Среднеквадратическое отклонение (стандартное отклонение)"""
    mu = meanvalue(data)
    variance = sum((x - mu) ** 2 for x in data) / len(data)
    return variance ** 0.5


def mean_absolute_error(actual, predicted):
    """Средняя абсолютная ошибка (MAE)"""
    n = len(actual)
    total_error = sum(abs(actual[i] - predicted[i]) for i in range(n))
    return total_error / n


def correlation(dataset1, dataset2):
    """Коэффициент корреляции Пирсона (от -1 до +1)"""
    n = len(dataset1)
    mu1 = sum(dataset1) / n
    mu2 = sum(dataset2) / n
    numerator = 0
    sum_sq_diff1 = 0
    sum_sq_diff2 = 0
    for i in range(n):
        diff1 = dataset1[i] - mu1
        diff2 = dataset2[i] - mu2
        numerator += diff1 * diff2
        sum_sq_diff1 += diff1 ** 2
        sum_sq_diff2 += diff2 ** 2
    denominator = (sum_sq_diff1 * sum_sq_diff2) ** 0.5
    return numerator / denominator if denominator != 0 else 0


print('Функции определены успешно')

## 2. Загрузка данных

In [ ]:
import pandas as pd

marks = pd.read_csv('EdStatsCountry.csv')

print('Колонки в датасете:')
print(marks.columns.tolist())
print(f'\nВсего строк: {len(marks)}, колонок: {len(marks.columns)}')

In [ ]:
# Первые 5 строк таблицы
marks.head()

## 3. Отбор числовых колонок

In [ ]:
# Оставляем только числовые колонки
numeric_marks = marks.select_dtypes(include=['number'])

# Убираем колонки с менее чем 10 непустыми значениями
numeric_marks = numeric_marks.dropna(axis=1, thresh=10)

# Убираем служебную пустую колонку
if 'Unnamed: 31' in numeric_marks.columns:
    numeric_marks = numeric_marks.drop(columns=['Unnamed: 31'])

cols = numeric_marks.columns.tolist()
print(f'Числовые колонки ({len(cols)}):')
for c in cols:
    print(f'  - {c}')

In [ ]:
# Краткая сводка по числовым колонкам
numeric_marks.describe()

## 4. Корреляции между парами числовых колонок

In [ ]:
print('=' * 70)
print('КОРРЕЛЯЦИИ МЕЖДУ ПАРАМИ КОЛОНОК')
print('(считаются только по строкам без NaN в обеих колонках)')
print('=' * 70)

for i in cols:
    for j in cols:
        if i < j:
            pair = numeric_marks[[i, j]].dropna()
            if len(pair) < 2:
                print(f'{i} vs {j}: недостаточно данных')
                continue
            d1 = pair[i].tolist()
            d2 = pair[j].tolist()
            result = correlation(d1, d2)
            print(f'\n{i}  vs  {j}')
            print(f'  Корреляция: {result:.4f}  (по {len(pair)} наблюдениям)')

## 5. Статистика по каждой числовой колонке

In [ ]:
col_labels = {
    'National accounts reference year': 'Базовый год национальных счетов',
    'Latest industrial data':           'Последние данные по промышленности',
    'Latest trade data':                'Последние данные по торговле',
}

print('=' * 70)
print('СТАТИСТИКА ПО КАЖДОЙ ЧИСЛОВОЙ КОЛОНКЕ')
print('=' * 70)

for col in cols:
    clean = numeric_marks[col].dropna().tolist()
    if len(clean) < 2:
        print(f'\nКолонка: {col}')
        print('  Недостаточно данных для анализа')
        continue

    avg_val = meanvalue(clean)
    dev_val = meandeviation(clean)
    baseline_predictions = [avg_val] * len(clean)
    mae_val = mean_absolute_error(clean, baseline_predictions)

    label = col_labels.get(col, col)
    print(f'\nКолонка: {label}')
    print(f'  - Непустых значений:   {len(clean)}')
    print(f'  - Среднее значение:    {avg_val:.2f}')
    print(f'  - Среднеквадр. откл.:  {dev_val:.2f}')
    print(f'  - Базовый MAE:         {mae_val:.2f}')
    print('-' * 50)

## 6. Анализ категориальных колонок

In [ ]:
categorical_cols = {
    'Region':           'Регион',
    'Income Group':     'Группа дохода',
    'Lending category': 'Категория кредитования',
    'System of trade':  'Система торговли',
}

print('=' * 70)
print('ТОП-3 ЗНАЧЕНИЙ В КАТЕГОРИАЛЬНЫХ КОЛОНКАХ')
print('=' * 70)

for col, label in categorical_cols.items():
    if col in marks.columns:
        top = marks[col].value_counts().head(3)
        print(f'\n{label} ({col}):')
        for val, cnt in top.items():
            print(f'  {val}: {cnt} стран')

## 7. Сводные таблицы

In [ ]:
# Последние данные по торговле — разбивка по регионам
if 'Region' in marks.columns and 'Latest trade data' in marks.columns:
    summary = marks.groupby('Region')['Latest trade data'].agg(
        Среднее='mean',
        Количество='count'
    ).dropna().sort_values('Среднее', ascending=False)
    print('Последние данные по торговле — по регионам:')
    display(summary)

In [ ]:
# Последние данные по торговле — разбивка по группам дохода
if 'Income Group' in marks.columns and 'Latest trade data' in marks.columns:
    summary2 = marks.groupby('Income Group')['Latest trade data'].agg(
        Среднее='mean',
        Количество='count'
    ).dropna().sort_values('Среднее', ascending=False)
    print('Последние данные по торговле — по группам дохода:')
    display(summary2)